# Unified Burned Area Prediction (Sentinel-2 + Landsat)

This script performs burned area prediction using satellite imagery from PlanetScope, augmented with either Sentinel-2 or Landsat data. It processes the input rasters, computes various indices (NDVI, NBR), loads a pre-trained UNet model, performs inference on image patches, reconstructs the full prediction mask, applies a nodata mask, saves the result as a GeoTIFF, and visualizes the output.

#⚙️ Configuration (set all the parameters)

In [ ]:
# ====================== CONFIG ======================
project_dir = "/content/drive/MyDrive/U-Net_burned_areas_project"  # Change to your project directory on Google Drive

# --- Main settings ---
image_dir_name = 'site-413-tropical-savanna'          # site folder
image_name     = '413_predictors.tif'                 # PlanetScope image

# --- Choose a model (check in the folder models/final/) ---
model_name = 'unet_dataset_PS_SENT_432_682_Ayman_best.pt'

#⏳ Prediction... (run all the cells)

In [ ]:
# json model metadata
json_name = model_name.replace('_best.pt', '.json')

# Paths to Sentinel-2 and Landsat
image_dir_for_paths = image_dir_name
input_base          = f"{project_dir}/User's_Inputs/Input_Rasters_For_Prediction/{image_dir_for_paths}"
image_path          = f"{input_base}/{image_name}"
path_for_prediction = f"{project_dir}/Predictions/{image_dir_name}"
model_dir           = f"{project_dir}/models/final/"

print(f"Model : {model_name}")
print(f"JSON  : {json_name}")
print(f"Input dir: {image_dir_for_paths}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd $project_dir

In [ ]:
# Set seed for reproducibility
SEED = 42

# Import necessary libraries
import os
import shutil
from pathlib import Path
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['MPLCONFIGDIR'] = os.getcwd() + '/configs/'

# Suppress warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Import necessary modules
import logging
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torch.nn import functional as F
from torchsummary import summary
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision.transforms import v2 as transforms
from torch.utils.data import TensorDataset, DataLoader, Dataset
import math
import matplotlib.gridspec as gridspec

!pip install torchview
from torchview import draw_graph

# Install torchmetrics for native metrics
!pip install -q torchmetrics
import torchmetrics
print(f"torchmetrics version: {torchmetrics.__version__}")

# Configure device and seeds
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    device = torch.device("cpu")


# Setup directories
logs_dir = "tensorboard"
!pkill -f tensorboard
%load_ext tensorboard
!mkdir -p models

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Configure plot display settings
sns.set(font_scale=1.4)
sns.set_style('white')
plt.rc('font', size=14)
%matplotlib inline

# Add functions for training, visualisation and model
from src.utils import (
    NPYSegmentationDataset,
    make_dataloaders,
    apply_colormap,
    plot_sample_images,
    UNetBlock,
    UNet,
    train_segmentation_model,
    plot_triptychs_from_loader,
    find_representative_indices,
    plot_triptychs_from_loader_with_selection,
    plot_layer_outputs_from_loader,
)

In [ ]:
import os
import sys
import json
import numpy as np
import torch
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
import matplotlib.pyplot as plt

# ====================== SETUP ======================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# === FIXED src/utils IMPORT ===
# Ensure project_dir is correctly set in CONFIG
sys.path.insert(0, project_dir)

from src.utils import UNet
print("✅ UNet successfully imported")

# ====================== LOAD METADATA ======================
metadata_path = os.path.join(model_dir, json_name)
with open(metadata_path, 'r') as f:
    meta = json.load(f)

NUM_CLASSES = meta.get('num_classes', 2)
IN_CHANNELS = meta.get('in_channels')
ps_bands    = meta.get('ps_bands', [])
s2_bands    = meta.get('s2_bands', [])
ls_bands    = meta.get('ls_bands', [])

# Automatically determine satellite type
if s2_bands:
    satellite_type = 'sentinel2'
    pre_path  = f"{input_base}/Sentinel_PRE.tif"
    post_path = f"{input_base}/Sentinel_POST.tif"
    NIR_BAND, RED_BAND, SWIR_BAND = 8, 4, 12
    resample_bands = s2_bands
elif ls_bands:
    satellite_type = 'landsat'
    pre_path  = f"{input_base}/Landsat_PRE.tif"
    post_path = f"{input_base}/Landsat_POST.tif"
    NIR_BAND, RED_BAND, SWIR_BAND = 5, 4, 7
    resample_bands = ls_bands
else:
    raise ValueError("Could not determine satellite type from metadata")

print(f"Detected satellite: {satellite_type.upper()}")

In [ ]:
# ====================== HELPER FUNCTIONS ======================
def compute_ndvi(nir, red):
    return (nir - red) / (nir + red + 1e-8)

def compute_nbr(nir, swir):
    return (nir - swir) / (nir + swir + 1e-8)

def normalize_patch(patch):
    out = np.empty_like(patch, dtype=np.float32)
    for c in range(patch.shape[0]):
        ch = np.nan_to_num(patch[c].astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        mn, mx = ch.min(), ch.max()
        out[c] = (ch - mn) / (mx - mn + 1e-8)
    return out

def resample_to_ref(src_path, band_idx, ref_profile):
    with rasterio.open(src_path) as src:
        data = src.read(band_idx).astype(np.float32)
        dst = np.zeros((ref_profile['height'], ref_profile['width']), dtype=np.float32)
        reproject(
            source=data, destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_profile['transform'], dst_crs=ref_profile['crs'],
            resampling=Resampling.cubic
        )
    return dst

In [ ]:
# ====================== PREPARE DATA ======================
os.makedirs(path_for_prediction, exist_ok=True)
patches_dir = os.path.join(path_for_prediction, "patches_tmp")
os.makedirs(patches_dir, exist_ok=True)

with rasterio.open(image_path) as src:
    original_H = src.height
    original_W = src.width
    profile = src.profile.copy()
    ref_profile = {
        'height': original_H,
        'width': original_W,
        'transform': src.transform,
        'crs': src.crs
    }

print(f"Image size: {original_H} x {original_W}")

# Resample bands + compute indices
print(f"\nResampling {satellite_type.upper()} bands...")
sat_pre_full = np.stack([resample_to_ref(pre_path, b, ref_profile) for b in resample_bands], axis=0)
sat_post_full = np.stack([resample_to_ref(post_path, b, ref_profile) for b in resample_bands], axis=0)

print("Computing indices...")
nir_pre = resample_to_ref(pre_path, NIR_BAND, ref_profile)
red_pre = resample_to_ref(pre_path, RED_BAND, ref_profile)
swir_pre = resample_to_ref(pre_path, SWIR_BAND, ref_profile)

nir_post = resample_to_ref(post_path, NIR_BAND, ref_profile)
red_post = resample_to_ref(post_path, RED_BAND, ref_profile)
swir_post = resample_to_ref(post_path, SWIR_BAND, ref_profile)

indices_full = np.stack([
    compute_ndvi(nir_pre, red_pre),
    compute_ndvi(nir_post, red_post),
    compute_nbr(nir_pre, swir_pre),
    compute_nbr(nir_post, swir_post),
    compute_ndvi(nir_post, red_post) - compute_ndvi(nir_pre, red_pre),
    compute_nbr(nir_pre, swir_pre) - compute_nbr(nir_post, swir_post)
], axis=0)

In [ ]:
# ====================== LOAD MODEL ======================
model = UNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES).to(device)
best_model_path = os.path.join(model_dir, model_name)
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Model loaded: {model_name}")

In [ ]:
# ====================== INFERENCE ======================
INFERENCE_PATCH_SIZE = 128
ps_rasterio_bands = ps_bands

row_starts = list(range(0, original_H, INFERENCE_PATCH_SIZE))
col_starts = list(range(0, original_W, INFERENCE_PATCH_SIZE))
total_patches = len(row_starts) * len(col_starts)

print(f"\nRunning inference on {total_patches} patches...")

with rasterio.open(image_path) as ps_src:
    with torch.no_grad():
        for idx, (row, col) in enumerate((r, c) for r in row_starts for c in col_starts):
            win_h = min(INFERENCE_PATCH_SIZE, original_H - row)
            win_w = min(INFERENCE_PATCH_SIZE, original_W - col)
            window = Window(col, row, win_w, win_h)

            ps_patch = ps_src.read(indexes=ps_rasterio_bands, window=window).astype(np.float32)
            sat_pre_patch  = sat_pre_full[:,  row:row+win_h, col:col+win_w]
            sat_post_patch = sat_post_full[:, row:row+win_h, col:col+win_w]
            idx_patch      = indices_full[:,  row:row+win_h, col:col+win_w]

            patch = np.concatenate([ps_patch, sat_pre_patch, sat_post_patch, idx_patch], axis=0)

            # Padding for edge patches
            if win_h < INFERENCE_PATCH_SIZE or win_w < INFERENCE_PATCH_SIZE:
                padded = np.zeros((IN_CHANNELS, INFERENCE_PATCH_SIZE, INFERENCE_PATCH_SIZE), dtype=np.float32)
                padded[:, :win_h, :win_w] = patch
                patch = padded

            patch = normalize_patch(patch)
            patch_tensor = torch.from_numpy(patch).unsqueeze(0).to(device)

            outputs = model(patch_tensor)
            pred = outputs.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.uint8)[:win_h, :win_w]

            np.save(os.path.join(patches_dir, f"patch_{idx:05d}_r{row}_c{col}.npy"), pred)

            if (idx + 1) % 50 == 0:
                print(f"  {idx+1}/{total_patches} patches done...")

print("Inference complete.")

In [ ]:
# ====================== RECONSTRUCT + NODATA MASK ======================
print("Reconstructing full mask...")
reconstructed = np.zeros((original_H, original_W), dtype=np.uint8)

for idx, (row, col) in enumerate((r, c) for r in row_starts for c in col_starts):
    win_h = min(INFERENCE_PATCH_SIZE, original_H - row)
    win_w = min(INFERENCE_PATCH_SIZE, original_W - col)
    patch_file = os.path.join(patches_dir, f"patch_{idx:05d}_r{row}_c{col}.npy")
    reconstructed[row:row+win_h, col:col+win_w] = np.load(patch_file)

# Apply nodata mask
with rasterio.open(image_path) as ps_src:
    ps_check = ps_src.read(indexes=ps_rasterio_bands[:4]).astype(np.float32)
nodata_mask = np.all(ps_check == 0, axis=0)
reconstructed[nodata_mask] = 255

print(f"Nodata pixels masked: {nodata_mask.sum()}")

In [ ]:
# ====================== SAVE GeoTIFF ======================
output_path = os.path.join(
    path_for_prediction,
    f"predicted_mask_{satellite_type}_{model_name.replace('_best.pt', '')}.tif"
)

out_profile = profile.copy()
out_profile.update(dtype=rasterio.uint8, count=1, compress='lzw', nodata=255)

with rasterio.open(output_path, 'w', **out_profile) as dst:
    dst.write(reconstructed, 1)

print(f"\n✅ Saved: {output_path}")

In [ ]:
# ====================== VISUALIZATION ======================
with rasterio.open(image_path) as src:
    factor = max(1, min(original_H, original_W) // 512)
    preview = src.read(
        indexes=ps_rasterio_bands[:3],
        out_shape=(3, original_H//factor, original_W//factor),
        resampling=rasterio.enums.Resampling.nearest
    ).astype(np.float32)

plt.figure(figsize=(15, 7))
plt.subplot(1, 2, 1)
rgb = np.transpose(preview, (1, 2, 0))
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
plt.imshow(rgb)
plt.title('PS RGB Preview')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(reconstructed[::factor, ::factor], cmap='viridis', vmin=0, vmax=1)
plt.title(f'Predicted Burned Area — {satellite_type.upper()}')
plt.colorbar(label='Class (0=Unburned, 1=Burned)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Optional cleanup
import shutil
shutil.rmtree(patches_dir)

##### Contributors:
- **Ayman Mutasim Alfadul Abdelgadir**: aymanmutasim@mail.polimi.it
- **Evgenii Miasnikov**: evgenii.miasnikov@mail.polimi.it

```
   Copyright 2026 Ayman Mutasim Alfadul Abdelgadir, Evgenii Miasnikov

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       https://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```